# EffoVPR-R standalone evaluation on Kaggle

Standalone continuation notebook for a completed EffoVPR-R fine-tuning run. It uses the **same embedded evaluation implementation and test split protocol as `EffoVPR_R.ipynb`** (seed 42; 70/10/10/10 split; duplicate-image groups kept together; `test_queries` evaluated against `gallery`).

Attach the EffoVPR model artifact as a Kaggle input. This notebook rebuilds the gallery index from that exact model, then evaluates it; do not attach the old gallery index. The source code is embedded below; no repository upload is required. The dataset is loaded from `PhamPhien/VN_Attractions` on Hugging Face, so enable Internet in Kaggle unless you also arrange a local dataset source.

Outputs are saved under `/kaggle/working/EffoVPR_R_Evaluation/artifacts/evaluation/effovpr-r/`.

## 1. Install dependencies

In [ ]:
%pip install -q transformers==5.18.0 datasets==5.0.1 faiss-cpu==1.15.1 "PyYAML>=6.0" "pandas>=2.0" "Pillow>=10.0" "matplotlib>=3.7" "tqdm>=4.9" "safetensors>=0.4.3"


## 2. Kaggle paths and input discovery

In [ ]:
import json, os, shutil, sys
from pathlib import Path

NOTEBOOK_NAME = "EffoVPR_R_Evaluation"
WORK_DIR = Path("/kaggle/working") / NOTEBOOK_NAME
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
for folder in ("src/effovpr/data", "src/effovpr/evaluation", "src/effovpr/models", "src/effovpr/reranking", "src/effovpr/retrieval", "src/effovpr/utils", "src/effovpr/visualization", "scripts", "configs", "artifacts"):
    (WORK_DIR / folder).mkdir(parents=True, exist_ok=True)
HF_CACHE = WORK_DIR / "artifacts/cache/huggingface"
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Exact Kaggle mount locations for the attached inputs shown in the notebook sidebar.
MODEL_DIR = Path("/kaggle/input/models/phinphm/effovpr-r-version1/pytorch/default/1")
if not MODEL_DIR.is_dir():
    raise FileNotFoundError(f"EffoVPR model input was not found: {MODEL_DIR}")
model_config_path = MODEL_DIR / "config.json"
model_weights_path = MODEL_DIR / "model.safetensors"
if not model_config_path.is_file() or not model_weights_path.is_file():
    raise FileNotFoundError(f"Model input must contain config.json and model.safetensors: {MODEL_DIR}")
model_config = json.loads(model_config_path.read_text(encoding="utf-8"))
if model_config.get("architecture") != "EffoVPR":
    raise ValueError(f"Unexpected model architecture in {model_config_path}: {model_config.get('architecture')!r}")
INDEX_DIR = WORK_DIR / "artifacts/index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)
# Remove any old bundle left in this Kaggle working directory so the new one
# is guaranteed to be built from the attached model below.
INDEX_FILES = ("gallery.index", "gallery_features.npy", "gallery_metadata.parquet", "index_config.json")
for filename in INDEX_FILES:
    (INDEX_DIR / filename).unlink(missing_ok=True)

DATASET_ID = "PhamPhien/VN_Attractions"
DATASET_SPLIT = "train"
MODE = "effovpr-r"
SEED = 42
BATCH_SIZE = 4
INFERENCE_RESOLUTION = 504
OUTPUT_DIR = WORK_DIR / "artifacts/evaluation" / MODE
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_DIR)
print("Gallery index: will be rebuilt from this model")
print("Dataset:", DATASET_ID, DATASET_SPLIT)
print("Workspace:", WORK_DIR)



## 3. Embedded source package
These source cells are taken from the corresponding module cells in `EffoVPR_R.ipynb`; they keep the split, model loading, retrieval, and MNN evaluation behavior aligned with that notebook.

In [ ]:
%%writefile src/__init__.py
"""EffoVPR package."""


In [ ]:
%%writefile src/effovpr/__init__.py
__all__ = ["EffoVPR", "CosFace", "build_effovpr_model"]

from .models.effovpr import EffoVPR
from .models.cosface import CosFace


def build_effovpr_model(*args, **kwargs):
    return EffoVPR(*args, **kwargs)


In [ ]:
%%writefile src/effovpr/data/audit.py
from __future__ import annotations

import json
import os
import hashlib
from collections import Counter
from typing import Any, Dict

import numpy as np
import pandas as pd
from PIL import Image


def compute_dataset_audit(dataset) -> Dict[str, Any]:
    ids = [str(item.get("id", "")) for item in dataset]
    labels = [str(item.get("label", "")) for item in dataset]
    invalid = 0
    resolutions = []
    modes = []
    image_hashes = []
    for item in dataset:
        image = item.get("image")
        if image is None:
            invalid += 1
            continue
        try:
            if not isinstance(image, Image.Image):
                if isinstance(image, dict) and image.get("bytes") is not None:
                    import io

                    image = Image.open(io.BytesIO(image["bytes"]))
                else:
                    image = Image.open(image)
            image = image.convert("RGB")
            modes.append(image.mode)
            resolutions.append((image.width, image.height))
            image_hashes.append(hashlib.sha256(image.tobytes()).hexdigest())
        except (OSError, TypeError, ValueError):
            invalid += 1
    counts = Counter(labels)
    stats = {
        "sample_count": len(dataset),
        "unique_ids": len(set(ids)),
        "unique_labels": len(set(labels)),
        "samples_per_label": dict(sorted(counts.items())),
        "minimum_samples_per_label": min(counts.values()) if counts else 0,
        "maximum_samples_per_label": max(counts.values()) if counts else 0,
        "mean_samples_per_label": float(np.mean(list(counts.values()))) if counts else 0.0,
        "median_samples_per_label": float(np.median(list(counts.values()))) if counts else 0.0,
        "invalid_image_count": invalid,
        "duplicate_id_count": len(ids) - len(set(ids)),
        "duplicate_image_count": len(image_hashes) - len(set(image_hashes)),
        "image_mode_statistics": dict(Counter(modes)),
        "image_resolution_statistics": {
            f"{width}x{height}": count
            for (width, height), count in sorted(Counter(resolutions).items())
        },
    }
    return stats


def save_dataset_audit(stats: Dict[str, Any], output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "dataset_audit.json"), "w", encoding="utf-8") as f:
        json.dump(stats, f, indent=2, ensure_ascii=False, sort_keys=True)
    df = pd.DataFrame([stats])
    df.to_csv(os.path.join(output_dir, "dataset_audit.csv"), index=False)


In [ ]:
%%writefile src/effovpr/data/dataset.py
from __future__ import annotations

import io
import os
from typing import Dict, Iterable, Optional

from pathlib import Path

_project_hub_cache = Path(__file__).resolve().parents[3] / "artifacts" / "cache" / "huggingface" / "hub"
os.environ.setdefault("HF_HUB_CACHE", str(_project_hub_cache))

import pandas as pd
from datasets import Dataset, load_dataset
from PIL import Image
from torch.utils.data import Dataset as TorchDataset


REQUIRED_COLUMNS = {"id", "image", "label"}


def load_vn_attractions(split: Optional[str] = "train", cache_dir: Optional[str] = None) -> Dataset:
    ds = load_dataset("PhamPhien/VN_Attractions", split=split, cache_dir=cache_dir)
    missing = REQUIRED_COLUMNS - set(ds.column_names)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")
    return ds


class VNAttractionsDataset(TorchDataset):
    def __init__(
        self,
        records: Iterable[dict],
        transform=None,
        image_processor=None,
        split: str = "train",
    ):
        self.records = list(records)
        self.transform = transform
        self.image_processor = image_processor
        self.split = split

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, idx: int):
        record = self.records[idx]
        image = record["image"]
        if image is None:
            raise ValueError(f"Missing image for record {record.get('id')}")
        if not isinstance(image, Image.Image):
            image = Image.open(io.BytesIO(image["bytes"])) if isinstance(image, dict) and "bytes" in image else Image.open(io.BytesIO(image))
        image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        label = record.get("label")
        return {
            "id": record.get("id"),
            "image": image,
            "label": label,
            "dataset_index": record.get("dataset_index", idx),
            "split": self.split,
        }


def build_label_mapping(dataset) -> Dict[str, int]:
    if hasattr(dataset, "__getitem__") and "label" in getattr(dataset, "column_names", []):
        labels = sorted({str(item) for item in dataset["label"]})
    else:
        labels = sorted({str(item.get("label", "")) for item in dataset})
    return {label: idx for idx, label in enumerate(labels)}


def dataset_to_dataframe(dataset: Dataset) -> pd.DataFrame:
    rows = []
    for i, item in enumerate(dataset):
        rows.append({
            "id": str(item.get("id", "")),
            "label": str(item.get("label", "")),
            "dataset_index": i,
        })
    return pd.DataFrame(rows)


In [ ]:
%%writefile src/effovpr/data/manifest.py
from __future__ import annotations

import json
import os
from typing import Any, Dict


def write_run_metadata(path: str, metadata: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, sort_keys=True)


In [ ]:
%%writefile src/effovpr/data/split.py
from __future__ import annotations

import os
import hashlib
import io
from collections import defaultdict
from typing import Dict

import numpy as np
import pandas as pd
from datasets import Dataset
from PIL import Image


SPLIT_NAMES = ("train", "gallery", "val_queries", "test_queries")


def _image_fingerprint(sample: dict) -> str | None:
    image = sample.get("image")
    if image is None:
        return None
    try:
        if isinstance(image, Image.Image):
            decoded = image.convert("RGB")
        elif isinstance(image, dict) and image.get("bytes") is not None:
            decoded = Image.open(io.BytesIO(image["bytes"])).convert("RGB")
        else:
            decoded = Image.open(io.BytesIO(image) if isinstance(image, bytes) else image).convert("RGB")
        return hashlib.sha256(decoded.tobytes()).hexdigest()
    except (OSError, TypeError, ValueError):
        return None


def deterministic_label_split(dataset: Dataset, seed: int = 42, train_ratio: float = 0.70, gallery_ratio: float = 0.10, val_ratio: float = 0.10, test_ratio: float = 0.10) -> Dict[str, pd.DataFrame]:
    ratios = (train_ratio, gallery_ratio, val_ratio, test_ratio)
    if any(ratio < 0 for ratio in ratios) or abs(sum(ratios) - 1.0) > 1e-6:
        raise ValueError("Split ratios must be non-negative and sum to 1.0")
    samples = []
    image_groups: dict[str, list[int]] = defaultdict(list)
    ids = set()
    for index, sample in enumerate(dataset):
        sample_id = str(sample.get("id", ""))
        label = sample.get("label")
        if not sample_id.strip():
            raise ValueError("Every dataset sample must have a non-empty id")
        if label is None:
            raise ValueError("Every dataset sample must have a label")
        if sample_id in ids:
            raise ValueError("Duplicate sample IDs prevent leakage-safe split generation")
        ids.add(sample_id)
        samples.append({"id": sample_id, "label": str(label)})
        fingerprint = _image_fingerprint(sample)
        if fingerprint is not None:
            image_groups[fingerprint].append(index)
    parent = list(range(len(samples)))

    def find(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    def union(left: int, right: int) -> None:
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parent[right_root] = left_root

    for indices in image_groups.values():
        for index in indices[1:]:
            union(indices[0], index)
    group_to_indices: dict[int, list[int]] = defaultdict(list)
    for index in range(len(samples)):
        group_to_indices[find(index)].append(index)

    label_to_groups: dict[str, list[list[int]]] = defaultdict(list)
    for group in group_to_indices.values():
        labels = {str(samples[index]["label"]) for index in group}
        # Cross-label duplicate images remain together; use a stable owner label
        # to decide their split without duplicating the group.
        owner = sorted(labels)[0]
        label_to_groups[owner].append(group)
    rng = np.random.default_rng(seed)
    split_rows: dict[str, list[dict]] = {name: [] for name in SPLIT_NAMES}
    # Balance whole duplicate groups within each label while honoring ratios.
    for label in sorted(label_to_groups):
        groups = label_to_groups[label]
        rng.shuffle(groups)
        total = sum(map(len, groups))
        desired = np.asarray(ratios, dtype=float) * total
        counts = np.zeros(len(SPLIT_NAMES), dtype=int)
        assigned = 0
        for group in groups:
            remaining = desired - counts
            split_index = int(np.argmax(remaining))
            split_name = SPLIT_NAMES[split_index]
            for index in group:
                sample = samples[index]
                split_rows[split_name].append({
                    "id": str(sample["id"]),
                    "label": str(sample["label"]),
                    "dataset_index": int(index),
                    "split": split_name,
                })
            counts[split_index] += len(group)
            assigned += len(group)
        if assigned != total:
            raise RuntimeError(f"Internal split accounting failed for label {label}")
    frames = {name: pd.DataFrame(rows, columns=["id", "label", "dataset_index", "split"]) for name, rows in split_rows.items()}
    validate_split_frames(frames)
    return frames


def validate_split_frames(frames: Dict[str, pd.DataFrame]) -> None:
    missing = set(SPLIT_NAMES) - set(frames)
    if missing:
        raise ValueError(f"Missing required split(s): {sorted(missing)}")
    ids = [set(frames[name]["id"].astype(str)) for name in SPLIT_NAMES]
    for left in range(len(ids)):
        for right in range(left + 1, len(ids)):
            overlap = ids[left] & ids[right]
            if overlap:
                raise ValueError(f"Sample ID leakage between {SPLIT_NAMES[left]} and {SPLIT_NAMES[right]}: {len(overlap)}")
    indices = [set(frames[name]["dataset_index"].astype(int)) for name in SPLIT_NAMES]
    for left in range(len(indices)):
        for right in range(left + 1, len(indices)):
            if indices[left] & indices[right]:
                raise ValueError(f"Dataset-index leakage between {SPLIT_NAMES[left]} and {SPLIT_NAMES[right]}")


def save_split_frames(frames: Dict[str, pd.DataFrame], output_dir: str) -> None:
    validate_split_frames(frames)
    os.makedirs(output_dir, exist_ok=True)
    for split_name, frame in frames.items():
        path = os.path.join(output_dir, f"{split_name}.csv")
        frame.to_csv(path, index=False)


In [ ]:
%%writefile src/effovpr/evaluation/evaluator.py
from __future__ import annotations

from typing import List

from .metrics import mean_reciprocal_rank, per_class_recall_at_k, recall_at_k


def evaluate_label_retrieval(query_labels: List[str], ranked_labels: List[List[str]], ks: List[int] = [1, 5, 10]):
    results = {}
    for k in ks:
        results[f"Recall@{k}"] = recall_at_k(query_labels, ranked_labels, k=k)
        per_class = per_class_recall_at_k(query_labels, ranked_labels, k=k)
        results[f"macro_Recall@{k}"] = float(sum(per_class.values()) / len(per_class)) if per_class else 0.0
        results[f"per_class_Recall@{k}"] = per_class
    results["MRR"] = mean_reciprocal_rank(query_labels, ranked_labels)
    return results


In [ ]:
%%writefile src/effovpr/evaluation/metrics.py
from __future__ import annotations

import numpy as np


def recall_at_k(labels: list[str], preds: list[str], k: int = 1):
    if len(labels) != len(preds):
        raise ValueError("labels and preds length mismatch")
    correct = 0
    for gt, topk in zip(labels, preds):
        topk_list = topk[:k]
        if gt in topk_list:
            correct += 1
    return correct / len(labels) if len(labels) else 0.0


def mean_reciprocal_rank(labels: list[str], preds: list[str]):
    if len(labels) != len(preds):
        raise ValueError("labels and preds length mismatch")
    rrs = []
    for gt, ranked in zip(labels, preds):
        for idx, item in enumerate(ranked, start=1):
            if item == gt:
                rrs.append(1.0 / idx)
                break
        else:
            rrs.append(0.0)
    return float(np.mean(rrs)) if rrs else 0.0


def per_class_recall_at_k(labels: list[str], preds: list[str], k: int = 1):
    if len(labels) != len(preds):
        raise ValueError("labels and preds length mismatch")
    grouped: dict[str, list[bool]] = {}
    for label, ranked in zip(labels, preds):
        grouped.setdefault(str(label), []).append(str(label) in [str(item) for item in ranked[:k]])
    return {label: float(np.mean(values)) for label, values in sorted(grouped.items())}


In [ ]:
%%writefile src/effovpr/evaluation/protocols.py
from __future__ import annotations

from typing import List


def label_based_correctness(query_label: str, gallery_labels: List[str]):
    return any(label == query_label for label in gallery_labels)


In [ ]:
%%writefile src/effovpr/models/cosface.py
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


class CosFace(nn.Module):
    def __init__(self, in_features: int, out_features: int, scale: float = 30.0, margin: float = 0.4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.scale = scale
        self.margin = margin
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings: torch.Tensor, targets: torch.Tensor):
        if targets is None:
            raise ValueError("CosFace requires target class indices")
        if embeddings.dim() != 2:
            embeddings = embeddings.view(embeddings.size(0), -1)
        if embeddings.size(1) != self.in_features:
            raise ValueError(f"Expected embedding dimension {self.in_features}, got {embeddings.size(1)}")
        if targets.ndim != 1 or targets.size(0) != embeddings.size(0):
            raise ValueError("targets must be a 1D tensor with one class index per embedding")
        if targets.dtype != torch.long:
            targets = targets.long()
        if targets.numel() and (targets.min() < 0 or targets.max() >= self.out_features):
            raise ValueError(f"targets must be in [0, {self.out_features})")
        norm_embeddings = F.normalize(embeddings, p=2, dim=1)
        norm_weight = F.normalize(self.weight, p=2, dim=1)
        cosine = norm_embeddings @ norm_weight.T
        target_mask = F.one_hot(targets, num_classes=self.out_features).to(dtype=cosine.dtype)
        logits = (cosine - self.margin * target_mask) * self.scale
        loss = F.cross_entropy(logits, targets)
        return logits, loss


In [ ]:
%%writefile src/effovpr/models/dino_attention.py
from __future__ import annotations

import math

import torch


def resolve_layer_index(layer, total_layers: int) -> int:
    if isinstance(layer, str):
        layer = layer.strip().lower()
        if layer.startswith("n-"):
            offset = int(layer.split("-")[-1])
            idx = total_layers - offset
        elif layer == "n":
            idx = total_layers - 1
        elif layer.startswith("n") and layer[1:].isdigit():
            idx = int(layer[1:]) - 1
        else:
            raise ValueError(f"Unsupported layer token: {layer}")
    else:
        idx = int(layer)
    if idx < 0:
        idx = total_layers + idx
    if idx < 0 or idx >= total_layers:
        raise ValueError(f"Layer index {idx} is out of range for total layers {total_layers}.")
    return idx


def parse_tokens(hidden_tokens: torch.Tensor):
    if hidden_tokens.ndim != 3:
        raise ValueError(f"Expected [batch, tokens, dim], got {tuple(hidden_tokens.shape)}")
    batch_size = hidden_tokens.size(0)
    cls_token = hidden_tokens[:, 0:1, :]
    register_tokens = hidden_tokens[:, 1:5, :] if hidden_tokens.size(1) > 5 else hidden_tokens[:, 1:1, :]
    patch_tokens = hidden_tokens[:, 5:, :]
    return cls_token, register_tokens, patch_tokens


def attention_patch_scores(q_patch: torch.Tensor, k_cls: torch.Tensor, temperature: float | None = None):
    if q_patch.ndim != 2:
        q_patch = q_patch.reshape(q_patch.size(0), -1)
    if temperature is None:
        temperature = math.sqrt(q_patch.size(-1))
    scores = (q_patch @ k_cls.reshape(-1)) / temperature
    return torch.softmax(scores, dim=-1)


def assert_patch_count(patch_tokens: torch.Tensor, image_resolution: int | None = None):
    patch_count = patch_tokens.shape[1]
    grid_h = int(round(math.sqrt(patch_count)))
    grid_w = patch_count // grid_h if grid_h > 0 else 0
    if image_resolution is not None:
        expected = (image_resolution // 14) ** 2
        if patch_count != expected:
            raise AssertionError(f"Expected patch count {expected} for resolution {image_resolution}, got {patch_count}")
    else:
        if grid_h * grid_w != patch_count:
            raise AssertionError(f"patch count mismatch: {patch_count} != {grid_h} * {grid_w}")


In [ ]:
%%writefile src/effovpr/models/dino_backbone.py
from __future__ import annotations


import torch
from transformers import AutoConfig, AutoImageProcessor, AutoModel


class DINOv2Backbone(torch.nn.Module):
    def __init__(
        self,
        model_name: str = "facebook/dinov2-with-registers-large",
        with_registers: bool = True,
        cache_dir: str | None = None,
        backbone_config: dict | None = None,
        processor_config: dict | None = None,
    ):
        super().__init__()
        self.model_name = model_name
        self.with_registers = with_registers
        if backbone_config is None:
            self.model = AutoModel.from_pretrained(model_name, cache_dir=cache_dir)
            self.image_processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=cache_dir)
        else:
            config = dict(backbone_config)
            model_type = config.pop("model_type")
            self.model = AutoModel.from_config(AutoConfig.for_model(model_type, **config))
            from transformers import image_processing_utils
            import transformers
            processor = dict(processor_config)
            processor_type = processor.pop("image_processor_type")
            processor_class = getattr(transformers, processor_type)
            if not issubclass(processor_class, image_processing_utils.BaseImageProcessor):
                raise ValueError(f"Unsupported image processor: {processor_type}")
            self.image_processor = processor_class.from_dict(processor)
        self.hidden_dim = self.model.config.hidden_size
        self.patch_size = self.model.config.patch_size
        self.num_layers = len(self.encoder_layers())

    def encoder_layers(self):
        encoder = getattr(self.model, "encoder", None)
        if encoder is not None:
            layers = getattr(encoder, "layer", getattr(encoder, "layers", None))
            if layers is not None:
                return layers
        vit = getattr(self.model, "vit", None)
        encoder = getattr(vit, "encoder", None)
        if encoder is not None:
            layers = getattr(encoder, "layers", getattr(encoder, "layer", None))
            if layers is not None:
                return layers
        raise RuntimeError(f"Unsupported DINOv2 encoder structure for {type(self.model).__name__}")

    def forward(self, pixel_values: torch.Tensor, **kwargs):
        return self.model(pixel_values=pixel_values, output_hidden_states=True, return_dict=True, **kwargs)

    def configure_trainable_layers(self, last_n_layers: int = 5):
        if last_n_layers < 0 or last_n_layers > self.num_layers:
            raise ValueError(f"last_n_layers must be between 0 and {self.num_layers}")
        for param in self.model.parameters():
            param.requires_grad = False
        layers = self.encoder_layers()
        for layer in list(layers)[-last_n_layers:] if last_n_layers else []:
            for param in layer.parameters():
                param.requires_grad = True


In [ ]:
%%writefile src/effovpr/models/effovpr.py
from __future__ import annotations

from typing import Dict, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F

from .cosface import CosFace
from .dino_attention import assert_patch_count, attention_patch_scores, parse_tokens, resolve_layer_index
from .dino_backbone import DINOv2Backbone
from .projection import GlobalProjection


class EffoVPR(nn.Module):
    def __init__(
        self,
        backbone_name: str = "facebook/dinov2-with-registers-large",
        global_dim: int = 1024,
        num_classes: Optional[int] = None,
        fine_tune_last_n_layers: int = 5,
        with_registers: bool = True,
        cosface_scale: float = 30.0,
        cosface_margin: float = 0.4,
        cache_dir: str | None = None,
        backbone_config: dict | None = None,
        processor_config: dict | None = None,
    ):
        super().__init__()
        self.backbone = DINOv2Backbone(model_name=backbone_name, with_registers=with_registers, cache_dir=cache_dir, backbone_config=backbone_config, processor_config=processor_config)
        self.global_dim = global_dim
        self.global_projection = GlobalProjection(input_dim=self.backbone.hidden_dim, output_dim=global_dim)
        self.num_classes = num_classes
        self.cosface = CosFace(in_features=global_dim, out_features=num_classes, scale=cosface_scale, margin=cosface_margin) if num_classes is not None else None
        self.configure_trainable_layers(last_n_layers=fine_tune_last_n_layers)

    def configure_trainable_layers(self, last_n_layers: int = 5):
        self.fine_tune_last_n_layers = last_n_layers
        if last_n_layers < 0 or last_n_layers > self.backbone.num_layers:
            raise ValueError(f"last_n_layers must be between 0 and {self.backbone.num_layers}")
        self.backbone.configure_trainable_layers(last_n_layers=last_n_layers)
        for param in self.global_projection.parameters():
            param.requires_grad = True
        if self.cosface is not None:
            for param in self.cosface.parameters():
                param.requires_grad = True

    def _prepare_images(self, images):
        if isinstance(images, torch.Tensor):
            return images
        if isinstance(images, list) and len(images) > 0 and isinstance(images[0], torch.Tensor):
            return torch.stack(images, dim=0)
        processed = self.backbone.image_processor(images=images, return_tensors="pt")
        return processed["pixel_values"]

    def _apply_projection(self, cls_token: torch.Tensor) -> torch.Tensor:
        feat = self.global_projection(cls_token)
        return F.normalize(feat, p=2, dim=-1)

    def extract_global(self, image):
        pixel_values = self._prepare_images(image)
        pixel_values = pixel_values.to(next(self.parameters()).device)
        outputs = self.backbone(pixel_values)
        hidden = outputs.last_hidden_state
        cls_token = hidden[:, 0, :]
        return self._apply_projection(cls_token)

    def extract_global_and_local(self, image, layer: int | str = "n-1"):
        pixel_values = self._prepare_images(image)
        pixel_values = pixel_values.to(next(self.parameters()).device)
        outputs = self.backbone(pixel_values)
        hidden_states = outputs.hidden_states
        layer_index = resolve_layer_index(layer, self.backbone.num_layers)
        selected = hidden_states[layer_index + 1]
        cls_token, register_tokens, patch_tokens = parse_tokens(selected)
        patch_tokens = patch_tokens.reshape(pixel_values.size(0), -1, patch_tokens.size(-1))
        assert_patch_count(patch_tokens, image_resolution=int(pixel_values.shape[-1]) if pixel_values.shape[-1] % self.backbone.patch_size == 0 else None)
        global_feat = self._apply_projection(outputs.last_hidden_state[:, 0, :])
        return global_feat, {"layer": layer_index, "cls_token": cls_token, "register_tokens": register_tokens, "patch_tokens": patch_tokens}

    def extract_qkv(self, image, layer: int | str = "n-1"):
        pixel_values = self._prepare_images(image)
        pixel_values = pixel_values.to(next(self.parameters()).device)
        outputs = self.backbone(pixel_values)
        hidden_states = outputs.hidden_states
        layer_index = resolve_layer_index(layer, self.backbone.num_layers)
        input_tokens = hidden_states[layer_index]
        block = self.backbone.encoder_layers()[layer_index]
        if hasattr(block, "norm1"):
            attention_input = block.norm1(input_tokens)
        elif hasattr(block, "layernorm_before"):
            attention_input = block.layernorm_before(input_tokens)
        else:
            raise RuntimeError(f"Cannot locate pre-attention normalization in {type(block).__name__}")
        attention = getattr(block, "attention", getattr(block, "self_attn", None))
        if attention is None:
            raise RuntimeError(f"Cannot locate attention module in {type(block).__name__}")
        # Hugging Face's DINOv2 implementations expose either q_proj/k_proj/v_proj
        # directly or query/key/value under an inner attention module.
        projections = [getattr(attention, name, None) for name in ("q_proj", "k_proj", "v_proj")]
        if all(projection is not None for projection in projections):
            q_proj, k_proj, v_proj = projections
        else:
            inner_attention = getattr(attention, "attention", attention)
            projections = [getattr(inner_attention, name, None) for name in ("query", "key", "value")]
            if not all(projection is not None for projection in projections):
                raise RuntimeError(f"Unsupported Q/K/V projection structure in {type(attention).__name__}")
            q_proj, k_proj, v_proj = projections
        q = q_proj(attention_input)
        k = k_proj(attention_input)
        v = v_proj(attention_input)
        cls_token, register_tokens, _ = parse_tokens(q)
        v_patch_tokens = parse_tokens(v)[2]
        assert_patch_count(v_patch_tokens, image_resolution=int(pixel_values.shape[-1]) if pixel_values.shape[-1] % self.backbone.patch_size == 0 else None)
        return {
            "q": q,
            "k": k,
            "v": v,
            "cls_token": cls_token,
            "register_tokens": register_tokens,
            "q_patch": parse_tokens(q)[2],
            "k_patch": parse_tokens(k)[2],
            "v_patch": parse_tokens(v)[2],
            "layer": layer_index,
        }

    def compute_attention_score(self, qkv: Dict[str, torch.Tensor], T1: float = 0.05):
        q_patch = qkv["q_patch"][0]
        k_cls = qkv["k"][0, 0, :]
        scores = attention_patch_scores(q_patch, k_cls)
        selected = scores > T1
        selected_v = qkv["v_patch"][0][selected]
        return scores, selected, selected_v

    def forward(self, images, labels=None, **kwargs):
        pixel_values = self._prepare_images(images)
        outputs = self.backbone(pixel_values)
        hidden = outputs.last_hidden_state
        cls_token = hidden[:, 0, :]
        global_feat = self._apply_projection(cls_token)
        if labels is not None and self.cosface is not None:
            logits, loss = self.cosface(global_feat, labels)
            return global_feat, logits, loss
        return global_feat


In [ ]:
%%writefile src/effovpr/models/projection.py
from __future__ import annotations

import torch
import torch.nn as nn


class GlobalProjection(nn.Module):
    def __init__(self, input_dim: int = 1024, output_dim: int = 1024):
        super().__init__()
        if output_dim not in (128, 256, 1024):
            raise ValueError("Supported global dimensions are 128, 256, and 1024.")
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.proj = nn.Linear(input_dim, output_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


In [ ]:
%%writefile src/effovpr/reranking/keypoint_selection.py
from __future__ import annotations

import torch


def select_attention_keypoints(scores: torch.Tensor, threshold: float = 0.05):
    if scores.ndim != 1:
        scores = scores.reshape(-1)
    mask = scores > threshold
    selected = scores[mask]
    return mask, selected


def summarize_keypoint_selection(scores: torch.Tensor):
    if scores.ndim != 1:
        scores = scores.reshape(-1)
    selected = scores[scores > 0.05]
    return {
        "average_selected_patches": float(selected.mean().item()) if selected.numel() > 0 else 0.0,
        "minimum_selected_patches": int(selected.numel() if selected.numel() > 0 else 0),
        "maximum_selected_patches": int(selected.numel()),
        "std_selected_patches": float(selected.std(unbiased=False).item()) if selected.numel() > 1 else 0.0,
    }


In [ ]:
%%writefile src/effovpr/reranking/local_features.py
from __future__ import annotations

import torch


def local_descriptors_from_v(v_patch: torch.Tensor):
    return v_patch


def compute_local_similarity(query_desc: torch.Tensor, candidate_desc: torch.Tensor):
    q = query_desc / (query_desc.norm(dim=-1, keepdim=True) + 1e-8)
    c = candidate_desc / (candidate_desc.norm(dim=-1, keepdim=True) + 1e-8)
    return q @ c.T


In [ ]:
%%writefile src/effovpr/reranking/mnn.py
from __future__ import annotations

import torch


def mutual_nearest_neighbor_matches(query_local: torch.Tensor, candidate_local: torch.Tensor, threshold: float = 0.65):
    query_local = torch.as_tensor(query_local, dtype=torch.float32)
    candidate_local = torch.as_tensor(candidate_local, dtype=torch.float32)
    if query_local.ndim != 2 or candidate_local.ndim != 2:
        raise ValueError("Query and candidate local descriptors must be 2D tensors.")
    q_norm = query_local / (query_local.norm(dim=1, keepdim=True) + 1e-8)
    c_norm = candidate_local / (candidate_local.norm(dim=1, keepdim=True) + 1e-8)
    similarity = q_norm @ c_norm.T
    if similarity.numel() == 0:
        return torch.empty((0, 2), dtype=torch.long)
    query_to_candidate = similarity.argmax(dim=1)
    candidate_to_query = similarity.argmax(dim=0)
    n_queries = query_local.size(0)
    query_ids = torch.arange(n_queries, device=similarity.device)
    mutual = candidate_to_query[query_to_candidate] == query_ids
    valid_scores = similarity[query_ids, query_to_candidate][mutual]
    keep = valid_scores > threshold
    retained = query_ids[mutual][keep]
    matched_candidates = query_to_candidate[mutual][keep]
    # return a pair list (query_index, candidate_index)
    if retained.numel() == 0:
        return torch.empty((0, 2), dtype=torch.long, device=similarity.device)
    return torch.stack([retained, matched_candidates], dim=1).long()


In [ ]:
%%writefile src/effovpr/retrieval/index.py
from __future__ import annotations

import json
import os

import faiss
import numpy as np
import pandas as pd


def build_faiss_index(features: np.ndarray, index_type: str = "IndexFlatIP"):
    if features.ndim != 2:
        raise ValueError(f"Expected 2D feature matrix, got {features.shape}")
    if features.shape[0] == 0 or features.shape[1] == 0:
        raise ValueError("Feature matrix must be non-empty")
    if not np.isfinite(features).all():
        raise ValueError("Feature matrix contains non-finite values")
    if index_type == "IndexFlatIP":
        index = faiss.IndexFlatIP(features.shape[1])
    else:
        raise ValueError(f"Unsupported index type: {index_type}")
    normalized = features.astype(np.float32, copy=False)
    norms = np.linalg.norm(normalized, axis=1)
    if np.any(np.abs(norms - 1.0) > 1e-3):
        raise ValueError("Features must be L2-normalized before IndexFlatIP insertion")
    index.add(normalized)
    return index


def search_faiss(index, query_features: np.ndarray, top_k: int = 10):
    if top_k <= 0:
        raise ValueError("top_k must be positive")
    if query_features.ndim != 2 or query_features.shape[1] != index.d:
        raise ValueError(f"Query features must have shape [N, {index.d}]")
    if not np.isfinite(query_features).all():
        raise ValueError("Query features contain non-finite values")
    query_features = query_features.astype(np.float32, copy=False)
    norms = np.linalg.norm(query_features, axis=1)
    if np.any(np.abs(norms - 1.0) > 1e-3):
        raise ValueError("Query features must be L2-normalized")
    distances, indices = index.search(query_features, min(top_k, index.ntotal))
    return distances, indices


def save_index_artifacts(index_path: str, features_path: str, metadata_path: str, config_path: str, index, features, metadata, config):
    os.makedirs(os.path.dirname(index_path), exist_ok=True)
    if index.ntotal != len(metadata) or index.ntotal != len(features):
        raise ValueError(f"Index, feature, and metadata counts differ: {index.ntotal}, {len(features)}, {len(metadata)}")
    faiss.write_index(index, index_path)
    np.save(features_path, features)
    metadata.to_parquet(metadata_path, index=False)
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2, sort_keys=True)


def load_index_artifacts(
    index_path: str,
    features_path: str,
    metadata_path: str,
    config_path: str,
    expected_config: dict | None = None,
):
    paths = tuple(map(os.fspath, (index_path, features_path, metadata_path, config_path)))
    if not all(os.path.isfile(path) for path in paths):
        return None

    index_path, features_path, metadata_path, config_path = paths
    index = faiss.read_index(index_path)
    features = np.load(features_path, allow_pickle=False)
    metadata = pd.read_parquet(metadata_path)
    with open(config_path, "r", encoding="utf-8") as config_file:
        config = json.load(config_file)

    if not isinstance(config, dict):
        raise ValueError("Index configuration must be a JSON object")
    if expected_config is not None:
        mismatches = {
            key: (config.get(key), expected_value)
            for key, expected_value in expected_config.items()
            if config.get(key) != expected_value
        }
        if mismatches:
            raise ValueError(f"Saved index configuration does not match expected configuration: {mismatches}")

    if features.ndim != 2:
        raise ValueError(f"Saved gallery features must be 2D, got {features.shape}")
    if index.ntotal != len(features) or index.ntotal != len(metadata):
        raise ValueError(
            "Saved index, feature, and metadata counts differ: "
            f"{index.ntotal}, {len(features)}, {len(metadata)}"
        )
    if features.shape[1] != index.d:
        raise ValueError(
            f"Saved feature dimension {features.shape[1]} does not match index dimension {index.d}"
        )
    if not np.isfinite(features).all():
        raise ValueError("Saved gallery features contain non-finite values")
    norms = np.linalg.norm(features.astype(np.float32, copy=False), axis=1)
    if np.any(np.abs(norms - 1.0) > 1e-3):
        raise ValueError("Saved gallery features are not L2-normalized")
    if config.get("count") != index.ntotal:
        raise ValueError("Saved index count does not match its configuration")
    if config.get("feature_dimension") != index.d:
        raise ValueError("Saved index dimension does not match its configuration")
    if config.get("index_type") != "IndexFlatIP":
        raise ValueError(f"Unsupported saved index type: {config.get('index_type')!r}")

    return index, features, metadata, config


In [ ]:
%%writefile src/effovpr/retrieval/search.py
from __future__ import annotations

import numpy as np

from .index import search_faiss


def global_retrieval(index, query_features: np.ndarray, top_k: int = 10):
    scores, ids = search_faiss(index, query_features, top_k=top_k)
    return scores, ids


In [ ]:
%%writefile src/effovpr/utils/checkpoint.py
from __future__ import annotations

import os
import tempfile
from typing import Any, Dict
from pathlib import Path

import torch


def save_checkpoint(path: str, state: Dict[str, Any]) -> None:
    destination = Path(path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    file_descriptor, temporary_path = tempfile.mkstemp(
        prefix=f".{destination.name}.",
        suffix=".tmp",
        dir=destination.parent,
    )
    os.close(file_descriptor)
    try:
        torch.save(state, temporary_path)
        os.replace(temporary_path, destination)
    except Exception:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
        raise


def load_checkpoint(path: str, map_location: str = "cpu") -> Dict[str, Any]:
    return torch.load(path, map_location=map_location, weights_only=True)


In [ ]:
%%writefile src/effovpr/utils/device.py
from __future__ import annotations

import torch


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def maybe_autocast(device: torch.device):
    if device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def nullcontext():
    from contextlib import nullcontext as _nullcontext
    return _nullcontext()


In [ ]:
%%writefile src/effovpr/utils/logging.py
import logging
from typing import Optional


def get_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))
        logger.addHandler(handler)
    logger.setLevel(level)
    logger.propagate = False
    return logger


In [ ]:
%%writefile src/effovpr/utils/seed.py
import random

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [ ]:
%%writefile src/effovpr/visualization/matching.py
from __future__ import annotations

import os
from typing import Any, Dict


def save_visualization_summary(path: str, summary: Dict[str, Any]) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(str(summary))


In [ ]:
%%writefile src/effovpr/utils/model_artifact.py
"""Complete, offline EffoVPR distribution artifacts; checkpoints remain training-only."""
from __future__ import annotations

import json
from pathlib import Path

import torch
import transformers
from PIL import Image
from safetensors.torch import load_file, save_file

from ..models.effovpr import EffoVPR
from ..models.dino_attention import attention_patch_scores
from ..reranking.mnn import mutual_nearest_neighbor_matches


def load_model_artifact(model_dir, device="cpu"):
    model_dir = Path(model_dir)
    config = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))
    if config["format_version"] != 1 or config["architecture"] != "EffoVPR":
        raise ValueError("Unsupported EffoVPR artifact format")
    model = EffoVPR(**config["model"], backbone_config=config["backbone_config"],
                    processor_config=config["processor_config"])
    if model.backbone.hidden_dim != config["hidden_dim"] or model.backbone.patch_size != config["patch_size"]:
        raise ValueError("Artifact architecture metadata mismatch")
    model.load_state_dict(load_file(str(model_dir / "model.safetensors")), strict=True)
    return model.to(device).eval(), config


def prepare_image(model, image, resolution):
    if isinstance(image, (str, Path)):
        with Image.open(image) as opened:
            image = opened.convert("RGB")
    if resolution <= 0 or resolution % model.backbone.patch_size:
        raise ValueError("Resolution must be a positive multiple of patch size")
    image = image.convert("RGB").resize((resolution, resolution), Image.Resampling.BICUBIC)
    return model.backbone.image_processor(images=[image], do_resize=False,
        do_center_crop=False, return_tensors="pt")["pixel_values"].to(next(model.parameters()).device)


@torch.inference_mode()
def verify_model_artifact(original, model_dir, image):
    restored, config = load_model_artifact(model_dir, next(original.parameters()).device)
    original.eval()
    pixels = prepare_image(original, image, config["inference"]["resolution"])
    restored_pixels = prepare_image(restored, image, config["inference"]["resolution"])
    torch.testing.assert_close(pixels, restored_pixels, rtol=0, atol=0)
    first = original.extract_global(pixels)
    second = restored.extract_global(restored_pixels)
    assert first.shape == second.shape == (1, config["model"]["global_dim"])
    torch.testing.assert_close(first, second, rtol=1e-5, atol=1e-6)
    report = {"global_shape": list(second.shape), "max_absolute_difference": float((first-second).abs().max()),
              "cosine_similarity": float(torch.nn.functional.cosine_similarity(first, second).item())}
    settings = config["reranking"]
    a = original.extract_qkv(pixels, settings["layer"])
    b = restored.extract_qkv(restored_pixels, settings["layer"])
    for key in ("q", "k", "v", "q_patch", "k_patch", "v_patch"):
        torch.testing.assert_close(a[key], b[key], rtol=1e-5, atol=1e-6)
    scores_a = attention_patch_scores(a["q_patch"][0], a["k"][0, 0])
    scores_b = attention_patch_scores(b["q_patch"][0], b["k"][0, 0])
    mask_a, mask_b = scores_a > settings["T1"], scores_b > settings["T1"]
    assert torch.equal(mask_a, mask_b)
    facet = settings["facet"].lower() + "_patch"
    torch.testing.assert_close(a[facet][0, mask_a], b[facet][0, mask_b], rtol=1e-5, atol=1e-6)
    candidate = a[facet][0]
    matches_a = mutual_nearest_neighbor_matches(a[facet][0, mask_a], candidate, settings["T2"])
    matches_b = mutual_nearest_neighbor_matches(b[facet][0, mask_b], candidate, settings["T2"])
    assert torch.equal(matches_a, matches_b)
    report.update(local_patch_shape=list(b[facet].shape), selected_local_count=int(mask_b.sum()),
                  local_max_absolute_difference=float((a[facet]-b[facet]).abs().max()), strict_load=True,
                  mnn_matches_equal=True)
    return report


def export_best_model(model, checkpoint_path, output_dir, configuration, verification_image):
    """Load the full best state, export every parameter/buffer, and verify reload."""
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    if "model_state_dict" not in checkpoint:
        raise ValueError("Export requires a full best model_state_dict; trainable-only checkpoints are not supported")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    config = {
        "format_version": 1, "architecture": "EffoVPR",
        "model": {"backbone_name": model.backbone.model_name, "global_dim": model.global_dim,
            "num_classes": model.num_classes, "fine_tune_last_n_layers": model.fine_tune_last_n_layers,
            "with_registers": model.backbone.with_registers,
            "cosface_scale": model.cosface.scale if model.cosface else 30.0,
            "cosface_margin": model.cosface.margin if model.cosface else 0.4},
        "backbone_config": model.backbone.model.config.to_dict(),
        "processor_config": model.backbone.image_processor.to_dict(),
        "hidden_dim": model.backbone.hidden_dim, "patch_size": model.backbone.patch_size,
        "inference": configuration["inference"], "reranking": configuration["reranking"],
        "class_mapping": checkpoint.get("class_mapping"),
        "source": {"checkpoint": Path(checkpoint_path).name,
                   "selection_metric": "minimum_mean_training_loss",
                   "best_training_loss": checkpoint.get("best_training_loss")},
        "transformers_version": transformers.__version__,
    }
    # clone also breaks shared storage, while retaining every state_dict entry.
    state = {name: tensor.detach().cpu().contiguous().clone() for name, tensor in checkpoint["model_state_dict"].items()}
    save_file(state, str(output_dir / "model.safetensors"), metadata={"format": "pt"})
    del state, checkpoint
    (output_dir / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    report = verify_model_artifact(model, output_dir, verification_image)
    (output_dir / "README.md").write_text(
        "# EffoVPR\n\nFull best-checkpoint weights, including frozen backbone, projection and classifier.\n"
        "Architecture lives in the project Python source. No pretrained download is needed.\n"
        f"Use transformers {transformers.__version__}, PyTorch and safetensors.\n\n"
        "```python\nfrom src.effovpr.utils.model_artifact import load_model_artifact, prepare_image\n"
        "import torch\nmodel, config = load_model_artifact('artifacts/model')\n"
        "pixels = prepare_image(model, '/path/query.jpg', config['inference']['resolution'])\n"
        "with torch.inference_mode():\n    embedding = model.extract_global(pixels)\n"
        "    qkv = model.extract_qkv(pixels, config['reranking']['layer'])\n```\n\n"
        "Use_Model.ipynb accepts query/gallery file paths, retrieves Top 10 with global cosine similarity, "
        "then reranks those candidates by local MNN count (ties retain global rank).\n"
        "Download config.json and model.safetensors as separate files; do not zip the model.\n\n"
        "Verification against best checkpoint:\n```json\n" + json.dumps(report, indent=2) + "\n```\n",
        encoding="utf-8")
    print(json.dumps({"model_dir": str(output_dir), "safetensors_exists": (output_dir / "model.safetensors").is_file(),
                      "verification": report}, indent=2))
    return report


In [ ]:
%%writefile scripts/build_gallery_index.py
from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

from src.effovpr.retrieval.index import build_faiss_index, save_index_artifacts
from scripts.common import (
    build_model,
    checkpoint_identifier,
    get_dataset,
    get_splits,
    load_config,
    load_or_extract_global_features,
    records_for_split,
    set_deterministic_seed,
    write_run_metadata,
)


def main() -> None:
    parser = argparse.ArgumentParser(description="Build a FAISS index over the gallery split.")
    parser.add_argument("--config", default="configs/effovpr_vn_attractions.yaml")
    parser.add_argument("--checkpoint", default="artifacts/model")
    parser.add_argument("--batch-size", type=int)
    args = parser.parse_args()

    config = load_config(args.config)
    set_deterministic_seed(42)
    dataset = get_dataset(config)
    splits = get_splits(dataset, "artifacts/splits")
    gallery = records_for_split(dataset, splits["gallery"])
    if not gallery:
        raise ValueError("Gallery split is empty")
    checkpoint_path = args.checkpoint if Path(args.checkpoint).exists() else None
    if checkpoint_path is None:
        raise FileNotFoundError(f"Checkpoint does not exist: {args.checkpoint}")
    model = build_model(config, checkpoint_path=checkpoint_path)
    batch_size = args.batch_size or int(config["training"]["batch_size"])
    resolution = int(config["inference"]["resolution"])
    features = load_or_extract_global_features(
        model,
        gallery,
        resolution,
        batch_size,
        split="gallery",
        checkpoint_path=checkpoint_path,
        config=config,
    )
    index = build_faiss_index(features, config["retrieval"]["index_type"])
    metadata = pd.DataFrame(
        [
            {
                "image_id": record["id"],
                "label": record["label"],
                "dataset_index": record["dataset_index"],
                "global_feature_dimension": int(features.shape[1]),
                "checkpoint_identifier": checkpoint_identifier(checkpoint_path),
                "input_resolution": resolution,
                "model_identifier": config["model"]["backbone"],
            }
            for record in gallery
        ]
    )
    output = Path("artifacts/index")
    save_index_artifacts(
        str(output / "gallery.index"),
        str(output / "gallery_features.npy"),
        str(output / "gallery_metadata.parquet"),
        str(output / "index_config.json"),
        index,
        features,
        metadata,
        {
            "index_type": config["retrieval"]["index_type"],
            "normalized": True,
            "count": int(index.ntotal),
            "feature_dimension": int(features.shape[1]),
            "checkpoint_identifier": checkpoint_identifier(checkpoint_path),
            "input_resolution": resolution,
            "model_identifier": config["model"]["backbone"],
        },
    )
    write_run_metadata("artifacts/run_metadata.json", config, checkpoint_path, 42)
    print(f"Saved {index.ntotal} gallery vectors ({features.shape[1]}D) to {output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/common.py
from __future__ import annotations

import json
import hashlib
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image

from src.effovpr.data.dataset import load_vn_attractions
from src.effovpr.data.audit import compute_dataset_audit, save_dataset_audit
from src.effovpr.data.split import deterministic_label_split, save_split_frames, validate_split_frames
from src.effovpr.models.dino_attention import attention_patch_scores
from src.effovpr.models.effovpr import EffoVPR
from src.effovpr.utils.checkpoint import load_checkpoint
from src.effovpr.utils.device import get_device
from src.effovpr.utils.seed import set_seed


def load_config(path: str | Path) -> dict[str, Any]:
    with open(path, "r", encoding="utf-8") as stream:
        config = yaml.safe_load(stream)
    if not isinstance(config, dict):
        raise ValueError(f"Configuration at {path} must contain a YAML mapping")
    return config


def get_dataset(config: dict[str, Any]):
    dataset_config = config.get("dataset", {})
    return load_vn_attractions(
        split=dataset_config.get("source_split", "train"),
        cache_dir=dataset_config.get("cache_dir"),
    )


def ensure_dataset_audit(dataset, output_dir: str | Path = "artifacts") -> dict[str, Any]:
    audit_path = Path(output_dir) / "dataset_audit.json"
    ids = [str(image_id) for image_id in dataset["id"]]
    id_digest = hashlib.sha256("\n".join(ids).encode("utf-8")).hexdigest()
    if audit_path.is_file():
        existing = json.loads(audit_path.read_text(encoding="utf-8"))
        if existing.get("sample_count") == len(dataset) and existing.get("id_sha256") == id_digest:
            return existing
    stats = compute_dataset_audit(dataset)
    save_dataset_audit(stats, str(output_dir))
    return stats


def get_splits(dataset, split_dir: str | Path, seed: int = 42) -> dict[str, pd.DataFrame]:
    split_path = Path(split_dir)
    names = ("train", "gallery", "val_queries", "test_queries")
    paths = {name: split_path / f"{name}.csv" for name in names}
    manifest_path = split_path / "split_config.json"
    dataset_identity = hashlib.sha256("\n".join(map(str, dataset["id"])).encode("utf-8")).hexdigest()
    expected_manifest = {
        "seed": seed,
        "sample_count": len(dataset),
        "id_sha256": dataset_identity,
        "ratios": {"train": 0.7, "gallery": 0.1, "val_queries": 0.1, "test_queries": 0.1},
    }
    manifest = None
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    manifest_matches = bool(
        manifest is not None
        and all(manifest.get(key) == value for key, value in expected_manifest.items())
        and manifest.get("split_csv_sha256")
        == {
            name: hashlib.sha256(path.read_bytes()).hexdigest()
            for name, path in paths.items()
            if path.is_file()
        }
    )
    if not all(path.is_file() for path in paths.values()) or not manifest_matches:
        frames = deterministic_label_split(dataset, seed=seed)
        save_split_frames(frames, split_path)
        manifest = {
            **expected_manifest,
            "split_csv_sha256": {
                name: hashlib.sha256(path.read_bytes()).hexdigest()
                for name, path in paths.items()
            },
        }
        manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding="utf-8")
    else:
        frames = {name: pd.read_csv(path, dtype={"id": str, "label": str}) for name, path in paths.items()}
        validate_split_frames(frames)
        for name, frame in frames.items():
            if not {"id", "label", "dataset_index", "split"}.issubset(frame.columns):
                raise ValueError(f"Split file {paths[name]} is missing required columns")
            if not frame.empty and (frame["dataset_index"].min() < 0 or frame["dataset_index"].max() >= len(dataset)):
                raise ValueError(f"Split file {paths[name]} contains out-of-range dataset indices")
    return frames


def records_for_split_metadata(dataset, frame: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    for row in frame.itertuples(index=False):
        sample = dataset[int(row.dataset_index)]
        if str(sample.get("id", "")) != str(row.id) or str(sample.get("label", "")) != str(row.label):
            raise ValueError(f"Split metadata does not match dataset row {row.dataset_index}")
        records.append({
            "id": str(row.id),
            "label": str(row.label),
            "dataset_index": int(row.dataset_index),
        })
    return records


def records_for_split(dataset, frame: pd.DataFrame) -> list[dict[str, Any]]:
    records = records_for_split_metadata(dataset, frame)
    for record in records:
        record["image"] = dataset[record["dataset_index"]]["image"]
    return records


def build_model(config: dict[str, Any], class_count: int | None = None, checkpoint_path: str | None = None, zero_shot: bool = False) -> EffoVPR:
    if checkpoint_path and Path(checkpoint_path).is_dir():
        from src.effovpr.utils.model_artifact import load_model_artifact
        model, _ = load_model_artifact(checkpoint_path, get_device())
        return model
    model_config = config["model"]
    model = EffoVPR(
        backbone_name=model_config["backbone"],
        global_dim=int(model_config.get("global_dim", 1024)),
        num_classes=None if zero_shot else class_count,
        fine_tune_last_n_layers=0 if zero_shot else int(model_config.get("fine_tune_last_n_layers", 5)),
        with_registers=bool(model_config.get("with_registers", True)),
        cosface_scale=float(model_config.get("cosface_scale", 30.0)),
        cosface_margin=float(model_config.get("cosface_margin", 0.4)),
        cache_dir=model_config.get("cache_dir"),
    )
    if checkpoint_path:
        checkpoint = load_checkpoint(checkpoint_path, map_location="cpu")
        state = checkpoint.get("model_state_dict", checkpoint.get("model_state"))
        if state is None:
            raise ValueError(f"Checkpoint {checkpoint_path} does not contain model weights")
        model.load_state_dict(state, strict=True)
    return model.to(get_device())


def image_batch(model: EffoVPR, images: list[Image.Image], resolution: int) -> torch.Tensor:
    if resolution <= 0 or resolution % model.backbone.patch_size:
        raise ValueError(f"Resolution must be a positive multiple of patch size {model.backbone.patch_size}")
    resized_images = [
        image.convert("RGB").resize(
            (resolution, resolution),
            resample=Image.Resampling.BICUBIC,
        )
        for image in images
    ]
    processed = model.backbone.image_processor(
        images=resized_images,
        do_resize=False,
        do_center_crop=False,
        return_tensors="pt",
    )
    return processed["pixel_values"].to(next(model.parameters()).device)


@torch.inference_mode()
def extract_global_features(
    model: EffoVPR,
    records: list[dict[str, Any]],
    resolution: int,
    batch_size: int,
    zero_shot: bool = False,
) -> np.ndarray:
    features: list[np.ndarray] = []
    model.eval()
    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]
        if any(record.get("image") is None for record in batch):
            raise ValueError("Missing image while extracting global features")
        pixels = image_batch(model, [record["image"] for record in batch], resolution)
        if zero_shot:
            output = model.backbone(pixels)
            descriptor = output.last_hidden_state[:, 0, :]
            descriptor = torch.nn.functional.normalize(descriptor, p=2, dim=-1)
        else:
            descriptor = model.extract_global(pixels)
        features.append(descriptor.float().cpu().numpy())
    if not features:
        return np.empty((0, model.backbone.hidden_dim if zero_shot else model.global_dim), dtype=np.float32)
    return np.concatenate(features).astype(np.float32, copy=False)


def load_or_extract_global_features(
    model: EffoVPR,
    records: list[dict[str, Any]],
    resolution: int,
    batch_size: int,
    split: str,
    checkpoint_path: str | None,
    config: dict[str, Any],
    zero_shot: bool = False,
) -> np.ndarray:
    checkpoint = Path(checkpoint_path).resolve() if checkpoint_path else None
    if checkpoint and checkpoint.is_dir():
        checkpoint = checkpoint / "model.safetensors"
    checkpoint_stat = checkpoint.stat() if checkpoint and checkpoint.is_file() else None
    metadata = {
        "checkpoint": str(checkpoint) if checkpoint else "pretrained-zero-shot",
        "checkpoint_size": checkpoint_stat.st_size if checkpoint_stat else None,
        "checkpoint_mtime_ns": checkpoint_stat.st_mtime_ns if checkpoint_stat else None,
        "model_identifier": config["model"]["backbone"],
        "input_resolution": int(resolution),
        "feature_dimension": model.backbone.hidden_dim if zero_shot else model.global_dim,
        "local_feature_layer": config["reranking"]["zero_shot_layer"] if zero_shot else config["reranking"]["layer"],
        "facet": config["reranking"]["facet"],
        "T1": config["reranking"]["T1"],
        "T2": config["reranking"]["T2"],
        "split": split,
        "record_ids_sha256": hashlib.sha256(
            "\n".join(str(record["id"]) for record in records).encode("utf-8")
        ).hexdigest(),
        "zero_shot": bool(zero_shot),
        "model_revision": getattr(model.backbone.model.config, "_commit_hash", None),
    }
    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode("utf-8")).hexdigest()[:20]
    cache_dir = Path("artifacts/features/global")
    cache_dir.mkdir(parents=True, exist_ok=True)
    features_path = cache_dir / f"{split}_{key}.npy"
    metadata_path = cache_dir / f"{split}_{key}.json"
    if features_path.is_file() and metadata_path.is_file():
        cached_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if cached_metadata != metadata:
            raise ValueError(f"Global feature cache metadata mismatch: {metadata_path}")
        features = np.load(features_path, allow_pickle=False)
        expected_shape = (len(records), int(metadata["feature_dimension"]))
        if features.shape != expected_shape or not np.isfinite(features).all():
            raise ValueError(f"Global feature cache has invalid shape or values: {features_path}")
        if len(features) and not np.allclose(np.linalg.norm(features, axis=1), 1.0, atol=1e-3):
            raise ValueError(f"Global feature cache contains non-normalized vectors: {features_path}")
        return features.astype(np.float32, copy=False)
    features = extract_global_features(model, records, resolution, batch_size, zero_shot=zero_shot)
    if features.shape != (len(records), int(metadata["feature_dimension"])):
        raise ValueError(f"Feature extractor returned unexpected shape {features.shape}")
    np.save(features_path, features)
    metadata_path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    return features


@torch.inference_mode()
def extract_local_descriptors(
    model: EffoVPR,
    image: Image.Image,
    resolution: int,
    layer: int | str,
    facet: str,
    threshold: float,
) -> torch.Tensor:
    facet_key = facet.lower()
    if facet_key not in {"q", "k", "v"}:
        raise ValueError("facet must be one of Q, K, or V")
    pixels = image_batch(model, [image], resolution)
    qkv = model.extract_qkv(pixels, layer=layer)
    scores = attention_patch_scores(qkv["q_patch"][0], qkv["k"][0, 0])
    keep = scores > threshold
    return qkv[f"{facet_key}_patch"][0, keep].float()


def checkpoint_identifier(path: str | None) -> str:
    return str(Path(path).resolve()) if path else "pretrained-zero-shot"


def write_run_metadata(path: str | Path, config: dict[str, Any], checkpoint: str | None, seed: int) -> dict[str, Any]:
    device = get_device()
    workspace = Path(__file__).resolve().parents[1]
    git_result = subprocess.run(
        ["git", "rev-parse", "HEAD"],
        cwd=workspace,
        capture_output=True,
        text=True,
        check=False,
    )
    metadata = {
        "seed": seed,
        "python_version": platform.python_version(),
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "dataset": config.get("dataset", {}).get("identifier", "PhamPhien/VN_Attractions"),
        "dataset_revision": None,
        "backbone": config["model"]["backbone"],
        "checkpoint": checkpoint_identifier(checkpoint),
        "input_resolution": config.get("inference", {}).get("resolution"),
        "training_resolution": config.get("training", {}).get("training_resolution"),
        "global_dimension": config.get("model", {}).get("global_dim"),
        "rerank_k": config.get("reranking", {}).get("top_k"),
        "local_feature_layer": config.get("reranking", {}).get("layer"),
        "facet": config.get("reranking", {}).get("facet"),
        "T1": config.get("reranking", {}).get("T1"),
        "T2": config.get("reranking", {}).get("T2"),
        "batch_size": config.get("training", {}).get("batch_size"),
        "git_commit": git_result.stdout.strip() if git_result.returncode == 0 else None,
        "configuration": config,
        "device": str(device),
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(metadata, indent=2, sort_keys=True), encoding="utf-8")
    return metadata


def ensure_parent(path: str | Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def set_deterministic_seed(seed: int) -> None:
    set_seed(seed)
    os.environ.setdefault("PYTHONHASHSEED", str(seed))


In [ ]:
%%writefile scripts/evaluate.py
from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.effovpr.evaluation.evaluator import evaluate_label_retrieval
from src.effovpr.retrieval.index import build_faiss_index, load_index_artifacts, search_faiss
from src.effovpr.reranking.mnn import mutual_nearest_neighbor_matches
from scripts.common import (
    build_model,
    checkpoint_identifier,
    extract_local_descriptors,
    get_dataset,
    get_splits,
    load_config,
    load_or_extract_global_features,
    records_for_split,
    set_deterministic_seed,
    write_run_metadata,
)


MODES = ("dino-zs", "effovpr-zs", "effovpr-g", "effovpr-r")


def main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate VN_Attractions label-based retrieval.")
    parser.add_argument("--config", default="configs/effovpr_vn_attractions.yaml")
    parser.add_argument("--checkpoint")
    parser.add_argument("--mode", choices=MODES, default="effovpr-r")
    parser.add_argument("--split", choices=("val_queries", "test_queries"), default="test_queries")
    parser.add_argument("--batch-size", type=int)
    parser.add_argument("--top-k", type=int)
    parser.add_argument("--layer")
    parser.add_argument("--facet", choices=("Q", "K", "V"))
    parser.add_argument("--t1", type=float)
    parser.add_argument("--t2", type=float)
    parser.add_argument("--resolution", type=int)
    parser.add_argument("--global-dim", type=int)
    parser.add_argument("--output-suffix", default="")
    parser.add_argument("--limit", type=int, help="Limit gallery and query counts for a smoke/limited experiment")
    args = parser.parse_args()

    config = load_config(args.config)
    if args.global_dim is not None:
        if args.global_dim not in {128, 256, 1024}:
            raise ValueError("--global-dim must be one of 128, 256, or 1024")
        config["model"]["global_dim"] = args.global_dim
    if args.resolution is not None:
        if args.resolution <= 0 or args.resolution % 14:
            raise ValueError("--resolution must be a positive multiple of the backbone patch size (14)")
        config["inference"]["resolution"] = args.resolution
    if args.layer is not None:
        config["reranking"]["layer"] = args.layer
        config["reranking"]["zero_shot_layer"] = args.layer
    if args.facet is not None:
        config["reranking"]["facet"] = args.facet
    if args.t1 is not None:
        config["reranking"]["T1"] = args.t1
    if args.t2 is not None:
        config["reranking"]["T2"] = args.t2
    if args.top_k is not None:
        config["reranking"]["top_k"] = args.top_k
    set_deterministic_seed(42)
    if args.limit is not None and args.limit <= 0:
        raise ValueError("--limit must be positive")
    zero_shot = args.mode in {"dino-zs", "effovpr-zs"}
    rerank = args.mode in {"effovpr-zs", "effovpr-r"}
    if not zero_shot and not args.checkpoint:
        raise ValueError(f"{args.mode} requires --checkpoint")
    if args.checkpoint and not Path(args.checkpoint).exists():
        raise FileNotFoundError(f"Checkpoint does not exist: {args.checkpoint}")

    dataset = get_dataset(config)
    splits = get_splits(dataset, "artifacts/splits")
    if args.limit is not None:
        gallery_frame = splits["gallery"].head(args.limit)
        gallery_labels = set(gallery_frame["label"].astype(str))
        query_frame = splits[args.split]
        positive_queries = query_frame[query_frame["label"].astype(str).isin(gallery_labels)]
        query_frame = (positive_queries if not positive_queries.empty else query_frame).head(args.limit)
    else:
        gallery_frame = splits["gallery"]
        query_frame = splits[args.split]
    gallery = records_for_split(dataset, gallery_frame)
    queries = records_for_split(dataset, query_frame)
    if not gallery or not queries:
        raise ValueError("Evaluation requires non-empty gallery and query splits")
    class_count = len({str(sample["label"]) for sample in dataset})
    model = build_model(
        config,
        class_count=class_count,
        checkpoint_path=args.checkpoint,
        zero_shot=zero_shot,
    )
    resolution = int(config["inference"]["resolution"])
    batch_size = args.batch_size or int(config["training"]["batch_size"])
    started = time.perf_counter()
    gallery_features = load_or_extract_global_features(
        model, gallery, resolution, batch_size, "gallery", args.checkpoint, config, zero_shot=zero_shot
    )
    query_features = load_or_extract_global_features(
        model, queries, resolution, batch_size, args.split, args.checkpoint, config, zero_shot=zero_shot
    )
    expected_index_config = {
        "index_type": config["retrieval"]["index_type"],
        "normalized": True,
        "count": len(gallery),
        "feature_dimension": int(gallery_features.shape[1]),
        "checkpoint_identifier": checkpoint_identifier(args.checkpoint),
        "input_resolution": resolution,
        "model_identifier": config["model"]["backbone"],
    }
    loaded_index = None
    if not zero_shot and args.limit is None:
        loaded_index = load_index_artifacts(
            "artifacts/index/gallery.index",
            "artifacts/index/gallery_features.npy",
            "artifacts/index/gallery_metadata.parquet",
            "artifacts/index/index_config.json",
            expected_index_config,
        )
    if loaded_index is None:
        index = build_faiss_index(gallery_features, config["retrieval"]["index_type"])
        index_source = "features cache (in-memory FAISS)"
    else:
        index, indexed_features, index_metadata, _ = loaded_index
        if index_metadata["image_id"].astype(str).tolist() != [record["id"] for record in gallery]:
            raise ValueError("Gallery metadata order does not match the deterministic gallery split")
        if index_metadata["label"].astype(str).tolist() != [record["label"] for record in gallery]:
            raise ValueError("Gallery metadata labels do not match the deterministic gallery split")
        if not np.allclose(indexed_features, gallery_features, rtol=1e-5, atol=1e-6):
            raise ValueError("Saved gallery index features differ from the active feature cache")
        index_source = "artifacts/index/gallery.index"
    requested_k = args.top_k or int(config["reranking"]["top_k"])
    if requested_k <= 0:
        raise ValueError("--top-k must be positive")
    candidate_k = min(max(requested_k, 100, 10), len(gallery))
    global_scores, global_indices = search_faiss(index, query_features, top_k=candidate_k)
    feature_extraction_and_search_seconds = time.perf_counter() - started

    reranked_indices: list[list[int]] = []
    reranked_scores: list[list[float]] = []
    rerank_seconds = 0.0
    if rerank:
        rerank_config = config["reranking"]
        local_layer = rerank_config["zero_shot_layer"] if zero_shot else rerank_config["layer"]
        facet = str(rerank_config["facet"]).upper()
        t1 = float(rerank_config["T1"])
        t2 = float(rerank_config["T2"])
        for query_index, query in enumerate(queries):
            rerank_start = time.perf_counter()
            query_local = extract_local_descriptors(
                model,
                query["image"],
                resolution,
                local_layer,
                facet,
                t1,
            )
            scored = []
            for original_rank, candidate_index in enumerate(global_indices[query_index][:requested_k]):
                candidate_index = int(candidate_index)
                if candidate_index < 0:
                    continue
                candidate_local = extract_local_descriptors(
                    model,
                    gallery[candidate_index]["image"],
                    resolution,
                    local_layer,
                    facet,
                    t1,
                )
                matches = mutual_nearest_neighbor_matches(query_local, candidate_local, threshold=t2)
                scored.append((candidate_index, float(len(matches)), original_rank, float(global_scores[query_index, original_rank])))
            scored.sort(key=lambda item: (-item[1], item[2]))
            reranked = [(item[0], item[1]) for item in scored]
            reranked_ids = {item[0] for item in scored}
            for original_rank, candidate_index in enumerate(global_indices[query_index]):
                candidate_index = int(candidate_index)
                if candidate_index >= 0 and candidate_index not in reranked_ids:
                    reranked.append((candidate_index, 0.0))
            reranked_indices.append([item[0] for item in reranked])
            reranked_scores.append([item[1] for item in reranked])
            rerank_seconds += time.perf_counter() - rerank_start
    else:
        reranked_indices = [
            [int(candidate) for candidate in row if candidate >= 0]
            for row in global_indices
        ]
        reranked_scores = [
            [float(score) for score, candidate in zip(score_row, id_row) if candidate >= 0]
            for score_row, id_row in zip(global_scores, global_indices)
        ]

    ranked_labels = [[gallery[index]["label"] for index in row] for row in reranked_indices]
    global_ranked_labels = [
        [gallery[int(index)]["label"] for index in row if index >= 0]
        for row in global_indices
    ]
    metrics = evaluate_label_retrieval(
        [query["label"] for query in queries],
        ranked_labels,
    )
    global_metrics = evaluate_label_retrieval(
        [query["label"] for query in queries],
        global_ranked_labels,
    )
    query_predictions = []
    failures = []
    for i, query in enumerate(queries):
        global_row = [int(candidate) for candidate in global_indices[i] if candidate >= 0]
        final_row = reranked_indices[i]
        global_correct_rank = next((rank for rank, idx in enumerate(global_row, 1) if gallery[idx]["label"] == query["label"]), None)
        final_correct_rank = next((rank for rank, idx in enumerate(final_row, 1) if gallery[idx]["label"] == query["label"]), None)
        global_top = global_row[0] if global_row else None
        final_top = final_row[0] if final_row else None
        row = {
            "query_id": query["id"],
            "query_label": query["label"],
            "global_rank_1_id": gallery[global_top]["id"] if global_top is not None else "",
            "global_rank_1_label": gallery[global_top]["label"] if global_top is not None else "",
            "global_rank_1_score": float(global_scores[i, 0]) if len(global_row) else None,
            "reranked_rank_1_id": gallery[final_top]["id"] if final_top is not None else "",
            "reranked_rank_1_label": gallery[final_top]["label"] if final_top is not None else "",
            "reranked_rank_1_score": reranked_scores[i][0] if reranked_scores[i] else None,
            "top_5_ids": [gallery[idx]["id"] for idx in final_row[:5]],
            "top_5_labels": [gallery[idx]["label"] for idx in final_row[:5]],
            "top_10_ids": [gallery[idx]["id"] for idx in final_row[:10]],
            "top_10_labels": [gallery[idx]["label"] for idx in final_row[:10]],
            "is_correct_at_1": bool(final_correct_rank == 1),
            "is_correct_at_5": bool(final_correct_rank is not None and final_correct_rank <= 5),
            "is_correct_at_10": bool(final_correct_rank is not None and final_correct_rank <= 10),
            "global_correct_rank": global_correct_rank,
            "reranked_correct_rank": final_correct_rank,
        }
        query_predictions.append(row)
        if global_correct_rank != 1 or final_correct_rank != 1:
            if global_correct_rank != 1 and final_correct_rank == 1:
                failure_category = "global wrong -> reranker correct"
            elif global_correct_rank == 1 and final_correct_rank != 1:
                failure_category = "global correct -> reranker wrong"
            elif (final_correct_rank is not None and final_correct_rank <= 10) or (
                global_correct_rank is not None and global_correct_rank <= 10
            ):
                failure_category = "correct result exists in top-10 but not top-1"
            elif (final_correct_rank is not None and final_correct_rank <= 100) or (
                global_correct_rank is not None and global_correct_rank <= 100
            ):
                failure_category = "correct result exists in top-100 but not top-10"
            else:
                failure_category = "correct result absent from top-100"
            failures.append({
                "query_id": query["id"],
                "query_label": query["label"],
                "global_prediction": row["global_rank_1_label"],
                "reranked_prediction": row["reranked_rank_1_label"],
                "global_correct_rank": global_correct_rank,
                "reranked_correct_rank": final_correct_rank,
                "failure_category": failure_category,
                "global_top_candidates": global_row[:10],
                "mnn_top_scores": reranked_scores[i][:10],
            })

    output_name = args.mode + (f"_{args.output_suffix}" if args.output_suffix else "")
    output = Path("artifacts/evaluation") / output_name
    output.mkdir(parents=True, exist_ok=True)
    result = {
        "status": "LIMITED EXPERIMENT" if args.split == "val_queries" or args.limit is not None else "FULL EXPERIMENT",
        "sample_limit": args.limit,
        "evaluation_protocol": "VN_Attractions Label-Based Retrieval",
        "mode": args.mode,
        "resolution": resolution,
        "reranking": {
            "top_k": requested_k,
            "layer": config["reranking"]["zero_shot_layer"] if zero_shot else config["reranking"]["layer"],
            "facet": config["reranking"]["facet"],
            "T1": config["reranking"]["T1"],
            "T2": config["reranking"]["T2"],
        },
        "query_split": args.split,
        "gallery_count": len(gallery),
        "query_count": len(queries),
        "metrics": metrics,
        "global_metrics": global_metrics,
        "global_feature_dimension": int(gallery_features.shape[1]),
        "retrieval_index": index_source,
        "gallery_feature_storage_bytes": int(gallery_features.nbytes),
        "global_extraction_and_search_seconds": feature_extraction_and_search_seconds,
        "reranking_seconds": rerank_seconds,
    }
    (output / "results.json").write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    pd.DataFrame(query_predictions).to_csv(output / "query_predictions.csv", index=False)
    failure_dir = Path("artifacts/failure_analysis") / output_name
    failure_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(failures).to_csv(failure_dir / "failure_cases.csv", index=False)
    write_run_metadata("artifacts/run_metadata.json", config, args.checkpoint, 42)
    print(json.dumps(result, indent=2, ensure_ascii=True))


if __name__ == "__main__":
    main()


## 4. Recreate the R configuration, rebuild the gallery index, and evaluate test_queries


In [ ]:
import yaml
CONFIG = {
 "model": {"backbone": "facebook/dinov2-with-registers-large", "global_dim": 1024, "fine_tune_last_n_layers": 5, "with_registers": True, "cosface_scale": 30.0, "cosface_margin": 0.4, "cache_dir": str(HF_CACHE / "models")},
 "dataset": {"identifier": DATASET_ID, "source_split": DATASET_SPLIT, "cache_dir": str(HF_CACHE / "datasets")},
 "training": {"batch_size": BATCH_SIZE, "epochs": 25, "training_resolution": 224, "backbone_lr": 1e-5, "classifier_lr": 1e-5, "weight_decay": 0.01, "mixed_precision": True, "workers": 2, "gradient_accumulation_steps": 1},
 "inference": {"resolution": INFERENCE_RESOLUTION},
 "reranking": {"enabled": True, "top_k": 100, "layer": "n-1", "zero_shot_layer": "n-2", "facet": "V", "T1": 0.05, "T2": 0.65},
 "retrieval": {"backend": "faiss", "index_type": "IndexFlatIP", "normalize": True},
 "evaluation": {"k_values": [1, 5, 10]},
}
CONFIG_PATH = WORK_DIR / "configs/effovpr_vn_attractions.yaml"
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False), encoding="utf-8")

import subprocess
PROJECT_ENV = os.environ.copy()
PROJECT_ENV["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + PROJECT_ENV.get("PYTHONPATH", "")
def run_stage(module, *args):
    command = [sys.executable, "-m", module, *map(str, args)]
    print("\n$ " + " ".join(command), flush=True)
    subprocess.run(command, cwd=WORK_DIR, env=PROJECT_ENV, check=True)

run_stage("scripts.build_gallery_index", "--config", CONFIG_PATH, "--checkpoint", MODEL_DIR)
print("New gallery index:", INDEX_DIR)

run_stage("scripts.evaluate", "--config", CONFIG_PATH, "--mode", MODE, "--split", "test_queries", "--resolution", INFERENCE_RESOLUTION, "--checkpoint", MODEL_DIR)
print("Evaluation complete:", OUTPUT_DIR)


## 5. Review results and create a downloadable package

The final cell creates a ZIP containing evaluation outputs, failure cases, the rebuilt gallery index, split metadata, and run configuration. Save a Kaggle version with output enabled, then download the ZIP from the notebook output or use the link displayed below.


In [ ]:
from IPython.display import Image as DisplayImage, display
RESULTS_PATH = OUTPUT_DIR / "results.json"
PREDICTIONS_PATH = OUTPUT_DIR / "query_predictions.csv"
if not RESULTS_PATH.is_file() or not PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(f"Expected evaluation outputs are missing in {OUTPUT_DIR}")
print(RESULTS_PATH.read_text(encoding="utf-8"))
print("Predictions:", PREDICTIONS_PATH)
for figure_path in sorted((WORK_DIR / "artifacts/visualizations" / MODE).glob("retrieval_*.png"))[:6]:
    print(figure_path.name)
    display(DisplayImage(filename=str(figure_path)))


from zipfile import ZIP_DEFLATED, ZipFile
from IPython.display import FileLink, display

OUTPUT_ZIP = Path("/kaggle/working/EffoVPR_R_Evaluation_outputs.zip")
PACKAGE_PATHS = [
    WORK_DIR / "artifacts/evaluation" / MODE,
    WORK_DIR / "artifacts/failure_analysis" / MODE,
    WORK_DIR / "artifacts/index",
    WORK_DIR / "artifacts/splits",
    WORK_DIR / "artifacts/visualizations" / MODE,
    WORK_DIR / "artifacts/run_metadata.json",
    WORK_DIR / "configs/effovpr_vn_attractions.yaml",
]
with ZipFile(OUTPUT_ZIP, "w", compression=ZIP_DEFLATED) as archive:
    for item in PACKAGE_PATHS:
        if item.is_file():
            archive.write(item, item.relative_to(WORK_DIR))
        elif item.is_dir():
            for file_path in sorted(item.rglob("*")):
                if file_path.is_file():
                    archive.write(file_path, file_path.relative_to(WORK_DIR))
print(f"Downloadable output package: {OUTPUT_ZIP} ({OUTPUT_ZIP.stat().st_size:,} bytes)")
display(FileLink(str(OUTPUT_ZIP)))
